# Task 1 — Fitting a diode I–V curve with an MLP

**ELEC-E8131 · AI for Electrical Engineers · Colab lab, part 1 of 4**

### What you will learn about Colab
- how cells, execution order and the *runtime* actually work
- how to check which hardware you were given, and how to switch it
- how to time a cell
- how to mount Google Drive and save results that survive a restart

### What you will learn about neural networks
- an MLP is a *function approximator*: here it approximates a device characteristic
- why a linear model cannot represent a diode
- why the **scaling of your target variable** decides whether training works at all
- what changes when you make the hidden layer wider

### Physics background
A p–n junction diode follows the Shockley equation

$$ I = I_S \left( e^{\,V_j / (n V_T)} - 1 \right) $$

with saturation current $I_S$, ideality factor $n$, junction voltage $V_j$, and thermal voltage
$V_T \approx 25.85$ mV at room temperature. A real device also has a series resistance $R_S$ from
the bulk semiconductor and the leads, so the voltage you measure at the terminals is

$$ V = V_j + I R_S $$

which is why a measured diode curve rolls off at high current instead of rising forever.

Two things make this an awkward regression problem: the current spans eight orders of magnitude,
and the curve is exponential at one end and resistive at the other.

---
## Part 0 — Execution order

A Colab notebook is **not** a script. Cells run in whatever order *you* click them, and they all
share one Python session. The number in `[ ]` on the left tells you the order things actually ran.

The four cells below are in the wrong order. **Do not reorder them yet.**

1. First run them top to bottom as they are. Write down the error you get and which cell produced it.
2. Now reorder them (drag with the handle at the top-right of each cell, or use the ↑ ↓ buttons)
   so that they run cleanly top to bottom.
3. Run them again.

In [ ]:
# Cell A
print("mean of y =", y.mean())

In [ ]:
# Cell B
y = np.sin(x)

In [ ]:
# Cell D
x = np.linspace(0, 2 * np.pi, 100)

In [ ]:
# Cell C
import numpy as np

### Now try this

With the cells in the correct order and everything working:

1. Change Cell B to `y = np.cos(x)` and run **only Cell B**. Then run **only Cell A**.
   Does Cell A print the mean of the sine or the cosine?
2. Go to **Runtime → Restart session**, then run **only Cell A**. What happens, and why?

> **The rule to remember:** the state of your notebook is the state of the Python session, not
> what the cells look like on screen. If something behaves strangely, *Runtime → Restart session
> and run all* is the honest way to check whether your notebook really works.

---
## Part 1 — What machine did you get?

Colab gives you a virtual machine. It may or may not have a GPU, and you do not choose which one.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("torch  :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))

In [ ]:
# The shell escape "!" runs a command on the Colab virtual machine, not in Python.
!nvidia-smi

If that printed an error about no NVIDIA driver, you are on a CPU runtime. Switch it on with
**Runtime → Change runtime type → T4 GPU**, then run the two cells above again.

⚠️ Changing the runtime type **restarts the session** and wipes every variable. Do it now, before
you have anything worth losing. We will come back to whether the GPU actually helps in Part 6.

---
## Part 2 — Make the measurement data

Instead of downloading a dataset we will simulate a measurement: sweep the bias voltage, evaluate
the Shockley equation, and add noise. Real curve tracers have multiplicative noise (a few percent
of the reading) rather than additive noise, so that is what we model.

In [ ]:
import pandas as pd

rng = np.random.default_rng(0)          # fixed seed => everyone gets the same data

I_S = 1e-9        # saturation current [A]
n_ideality = 1.5  # ideality factor [-]
V_T = 0.02585     # thermal voltage at ~300 K [V]
R_S = 2.0         # series resistance [ohm]

# Sweep the junction voltage, then work out what the terminal voltage would have been.
V_j = np.linspace(0.05, 0.75, 400)
I_true = I_S * (np.exp(V_j / (n_ideality * V_T)) - 1.0)        # Shockley equation [A]
V = V_j + I_true * R_S                                          # terminal voltage [V]
I_meas = I_true * np.exp(0.08 * rng.standard_normal(V.size))    # ~8 % multiplicative noise

df = pd.DataFrame({"V_volt": V, "I_amp": I_meas})
df.to_csv("diode_iv.csv", index=False)

print(df.head())
print()
print("current spans %.2e A to %.2e A  ->  %.1f decades"
      % (I_meas.min(), I_meas.max(), np.log10(I_meas.max() / I_meas.min())))

`diode_iv.csv` now sits in `/content/` on the virtual machine. Open the folder icon in the left
sidebar and find it.

**Important:** `/content/` is wiped when the runtime is recycled. Nothing you write there is
permanent. We fix that in Part 7.

In [ ]:
df = pd.read_csv("diode_iv.csv")
V = df["V_volt"].to_numpy()
I = df["I_amp"].to_numpy()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(V, I * 1e3, ".", ms=3)
ax[0].set_xlabel("V [V]"); ax[0].set_ylabel("I [mA]"); ax[0].set_title("linear scale")
ax[1].semilogy(V, I, ".", ms=3)
ax[1].set_xlabel("V [V]"); ax[1].set_ylabel("I [A]"); ax[1].set_title("log scale")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Question.** On the linear plot, roughly how much of the voltage range looks like "zero current"?
Look at the log plot: is the current actually zero there? Keep this in mind — it is the whole
difficulty of the task.

---
## Part 3 — A linear baseline

Before reaching for a neural network, always fit the simplest thing that could work.

In [ ]:
coef = np.polyfit(V, I, deg=1)
I_lin = np.polyval(coef, V)
rmse_lin = np.sqrt(np.mean((I_lin - I) ** 2))

plt.figure(figsize=(6, 4))
plt.plot(V, I * 1e3, ".", ms=3, label="measured")
plt.plot(V, I_lin * 1e3, "-", lw=2, label="linear fit")
plt.xlabel("V [V]"); plt.ylabel("I [mA]"); plt.legend(); plt.grid(alpha=.3)
plt.title("RMSE = %.2f mA" % (rmse_lin * 1e3))
plt.show()

print("linear fit predicts I(0.3 V) = %.3e A, truth is about %.3e A"
      % (np.polyval(coef, 0.3), np.interp(0.3, V, I_true)))

The linear fit is not merely inaccurate: it predicts **negative current** under forward bias over
part of the range, which is physically impossible. No amount of fitting effort repairs this,
because a straight line simply cannot be an exponential. This is what "the model class is wrong"
looks like.

---
## Part 4 — First MLP attempt, and why it fails

A multilayer perceptron with one input, two hidden layers of 32 `tanh` units, and one output.
With enough hidden units this can approximate any continuous function on a bounded interval —
in principle. Let us see what happens in practice.

First, a reusable training function. Read it before running it.

In [ ]:
def make_mlp(width=32, depth=2, n_in=1, n_out=1):
    layers, d = [], n_in
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.Tanh()]
        d = width
    layers += [nn.Linear(d, n_out)]
    return nn.Sequential(*layers)


def train_mlp(model, x, y, epochs=3000, lr=1e-2, verbose=True, seed=0):
    # x, y: torch tensors of shape (N, 1). Full-batch training - the dataset is tiny.
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        history.append(loss.item())
        if verbose and (epoch + 1) % (epochs // 5) == 0:
            print("epoch %5d   loss %.4e" % (epoch + 1, loss.item()))
    return np.array(history)

In [ ]:
# Inputs are always scaled. Voltage 0.05..0.75 -> roughly -1..1.
x_np = ((V - V.mean()) / V.std()).reshape(-1, 1)

# ATTEMPT 1: train directly on the current in amperes.
y_raw = I.reshape(-1, 1)

x_t = torch.tensor(x_np, dtype=torch.float32)
y_t = torch.tensor(y_raw, dtype=torch.float32)

torch.manual_seed(0)
model_raw = make_mlp(width=32, depth=2)
hist_raw = train_mlp(model_raw, x_t, y_t, epochs=3000, lr=1e-2)

In [ ]:
%%time
# Re-run the same training with %%time as the first line of the cell to see how long it takes.
torch.manual_seed(0)
model_raw = make_mlp(width=32, depth=2)
hist_raw = train_mlp(model_raw, x_t, y_t, epochs=3000, lr=1e-2, verbose=False)
print("final loss:", hist_raw[-1])

In [ ]:
with torch.no_grad():
    I_pred_raw = model_raw(x_t).numpy().ravel()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(V, I * 1e3, ".", ms=3, label="measured")
ax[0].plot(V, I_pred_raw * 1e3, "-", lw=2, label="MLP")
ax[0].set_xlabel("V [V]"); ax[0].set_ylabel("I [mA]"); ax[0].legend(); ax[0].set_title("linear scale: looks fine")

ax[1].semilogy(V, I, ".", ms=3, label="measured")
ax[1].semilogy(V, np.clip(I_pred_raw, 1e-12, None), "-", lw=2, label="MLP (clipped at 1e-12)")
ax[1].set_xlabel("V [V]"); ax[1].set_ylabel("I [A]"); ax[1].legend(); ax[1].set_title("log scale: the truth")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

frac_negative = np.mean(I_pred_raw < 0)
print("fraction of predictions that are negative: %.1f %%" % (100 * frac_negative))

### Why this happened

The mean squared error is dominated by the samples with the largest current. An error of 1 mA at
$V = 0.75$ V contributes as much to the loss as an error of 1 mA at $V = 0.2$ V — but at
$V = 0.2$ V the *true* current is around $10^{-6}$ A, so a 1 mA error there is wrong by three
orders of magnitude. The optimiser has no incentive to care.

The network is not broken. **The loss function is measuring the wrong thing**, because the target
variable spans eight decades and MSE is an absolute, not a relative, error measure.

---
## Part 5 — Fix it by changing the target

Two standard fixes:

- **Predict $\log_{10} I$ instead of $I$.** Errors then become relative errors, which is how you
  would judge a diode fit by eye on a semilog plot anyway.
- **Standardise the target** (subtract mean, divide by standard deviation) so it is $O(1)$, which
  keeps the gradients in a sensible range.

We do both. Note that we must remember the scaling constants in order to convert predictions back.

In [ ]:
y_log = np.log10(I).reshape(-1, 1)
y_mean, y_std = y_log.mean(), y_log.std()
y_scaled = (y_log - y_mean) / y_std

x_t = torch.tensor(x_np, dtype=torch.float32)
y_t = torch.tensor(y_scaled, dtype=torch.float32)

torch.manual_seed(0)
model_log = make_mlp(width=32, depth=2)
hist_log = train_mlp(model_log, x_t, y_t, epochs=3000, lr=1e-2)

In [ ]:
with torch.no_grad():
    y_hat_scaled = model_log(x_t).numpy().ravel()

log_I_pred = y_hat_scaled * y_std + y_mean     # undo standardisation
I_pred = 10.0 ** log_I_pred                    # undo the log

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(V, I, ".", ms=3, label="measured")
ax[0].semilogy(V, I_pred, "-", lw=2, label="MLP on log target")
ax[0].set_xlabel("V [V]"); ax[0].set_ylabel("I [A]"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title("now correct over all 8 decades")

ax[1].loglog(np.arange(1, len(hist_raw) + 1), hist_raw, label="raw target")
ax[1].loglog(np.arange(1, len(hist_log) + 1), hist_log, label="log target")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("training MSE"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title("loss curves")
plt.tight_layout(); plt.show()

rmse_decades = np.sqrt(np.mean((log_I_pred - np.log10(I)) ** 2))
print("MLP  RMSE in decades: %.4f  (typically within a factor of %.3f)"
      % (rmse_decades, 10 ** rmse_decades))

# Fair baseline: is the log-transformed problem now so easy that a straight line does it?
c_log = np.polyfit(V, np.log10(I), 1)
rmse_log_lin = np.sqrt(np.mean((np.polyval(c_log, V) - np.log10(I)) ** 2))
print("line RMSE in decades: %.4f" % rmse_log_lin)

Two things to take from the printout.

First, the MLP is now typically within about 8 % of the measured current across all eight decades
— which is the size of the measurement noise itself. You cannot do better than that, and you
should be suspicious of any model that appears to.

Second, look at the straight-line baseline on the log target. In the exponential region
$\log_{10} I$ really is close to linear in $V$, so the transform did most of the work — but the
series resistance bends the curve at high current, and that is the part only the network gets
right. **Changing the representation of the problem bought more than changing the model class
did.** That is the normal state of affairs in engineering machine learning.

> **Warning about loss curves.** The two curves above are *not comparable to each other* — they
> are MSE in different units. A loss curve only ever tells you whether optimisation is progressing.
> It never tells you whether your model is any good. For that you need a held-out set and a metric
> in physical units, like the RMSE in decades printed above.

---
## Part 6 — How wide should the hidden layer be?

Now hold out 25 % of the measurements, train on the rest, and evaluate on the held-out points.

The samples here are independent noisy draws around a smooth curve, so a random split is
legitimate. **This will not be true in Task 3** — remember that this is a property of the data,
not a universal rule.

In [ ]:
idx = rng.permutation(len(V))
n_test = len(V) // 4
test_idx, train_idx = idx[:n_test], idx[n_test:]

xtr = torch.tensor(x_np[train_idx], dtype=torch.float32)
ytr = torch.tensor(y_scaled[train_idx], dtype=torch.float32)
xte = torch.tensor(x_np[test_idx], dtype=torch.float32)
yte = torch.tensor(y_scaled[test_idx], dtype=torch.float32)

widths = [1, 2, 8, 64, 512]
results, curves = {}, {}

for w in widths:
    torch.manual_seed(0)
    m = make_mlp(width=w, depth=2)
    train_mlp(m, xtr, ytr, epochs=3000, lr=1e-2, verbose=False)
    with torch.no_grad():
        tr = torch.sqrt(((m(xtr) - ytr) ** 2).mean()).item() * y_std
        te = torch.sqrt(((m(xte) - yte) ** 2).mean()).item() * y_std
        curve = (m(torch.tensor(x_np, dtype=torch.float32)).numpy().ravel() * y_std + y_mean)
    results[w] = (tr, te)
    curves[w] = curve
    print("width %4d : train RMSE %.4f dec | test RMSE %.4f dec | %6d parameters"
          % (w, tr, te, sum(p.numel() for p in m.parameters())))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].semilogy(V, I, ".", ms=3, color="k", alpha=.4, label="measured")
for w in widths:
    ax[0].semilogy(V, 10.0 ** curves[w], "-", lw=1.6, label="width %d" % w)
ax[0].set_xlabel("V [V]"); ax[0].set_ylabel("I [A]"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].semilogx(widths, [results[w][0] for w in widths], "o-", label="train")
ax[1].semilogx(widths, [results[w][1] for w in widths], "s-", label="test")
ax[1].set_xlabel("hidden width"); ax[1].set_ylabel("RMSE [decades]"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**Read the numbers carefully — three different things are happening.**

1. **Width 1 underfits.** Too few units to bend the curve where it needs bending. Training error
   and test error are both high, and they are close to each other. That pair of symptoms *is* the
   definition of underfitting.
2. **Widths 8 and 64 are almost indistinguishable.** For a smooth one-dimensional function,
   capacity stops being the bottleneck very quickly. Roughly forty times more parameters buys
   essentially nothing.
3. **Width 512 is *worse*, and its training error is worse too.** If it were overfitting you would
   see low training error and high test error. You see neither. This is not a capacity problem —
   the same learning rate and epoch budget simply do not suit a much larger model. We return to
   this in Task 4, where you will find that learning rate matters more than architecture.

**Questions.**
- Where on the voltage axis does width 1 fail first, and does that location make physical sense?
- Sketch what the training and test curves would look like for a model that genuinely overfits.
  How would you tell that apart from case 3 above?

### Part 6b — What if you had taken fewer measurements? (optional)

Everything above used 400 measurement points. Curve tracers are fast, but plenty of engineering
data is expensive to collect. Repeat the sweep with only 40 points and watch what changes.

In [ ]:
sub = rng.permutation(len(V))[:40]
sub_test, sub_train = sub[:10], sub[10:]

xtr_s = torch.tensor(x_np[sub_train], dtype=torch.float32)
ytr_s = torch.tensor(y_scaled[sub_train], dtype=torch.float32)
xte_s = torch.tensor(x_np[sub_test], dtype=torch.float32)
yte_s = torch.tensor(y_scaled[sub_test], dtype=torch.float32)

for w in [1, 2, 8, 64]:
    torch.manual_seed(0)
    m = make_mlp(width=w, depth=2)
    train_mlp(m, xtr_s, ytr_s, epochs=3000, lr=1e-2, verbose=False)
    with torch.no_grad():
        tr = torch.sqrt(((m(xtr_s) - ytr_s) ** 2).mean()).item() * y_std
        te = torch.sqrt(((m(xte_s) - yte_s) ** 2).mean()).item() * y_std
    print("width %4d : train RMSE %.4f dec | test RMSE %.4f dec | gap %.4f"
          % (w, tr, te, te - tr))

Compare these numbers against the 400-point sweep above.

With 400 points, training and test error were almost the same — the model had no room to memorise
anything. With 30 training points, test error is roughly **double** the training error at every
width, and the absolute test error is about twice as bad as before. The architecture did not
change. Only the amount of data did.

Width 1 is worth a second look: it has both the highest error *and* the largest gap. It is
underfitting and it is being fitted to an unrepresentative handful of points at the same time.
Small datasets punish you twice.

The lesson to carry into Task 3: "how many parameters is too many" has no answer until you say how
much data you have, and how much of that data is genuinely independent.

---
## Part 7 — Does the GPU help?

Run this on a **GPU runtime** (Runtime → Change runtime type → T4 GPU). If you switch now you must
re-run the whole notebook, because restarting wipes all variables. *Runtime → Run all* does it.

In [ ]:
import time

def timed_training(device, width=32, epochs=3000):
    xd = torch.tensor(x_np, dtype=torch.float32).to(device)
    yd = torch.tensor(y_scaled, dtype=torch.float32).to(device)
    torch.manual_seed(0)
    m = make_mlp(width=width).to(device)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    train_mlp(m, xd, yd, epochs=epochs, lr=1e-2, verbose=False)
    if device == "cuda":
        torch.cuda.synchronize()
    return time.time() - t0

t_cpu = timed_training("cpu")
print("CPU: %.2f s" % t_cpu)
if torch.cuda.is_available():
    t_gpu = timed_training("cuda")
    print("GPU: %.2f s   (speed-up %.2fx)" % (t_gpu, t_cpu / t_gpu))
else:
    print("No GPU on this runtime - switch it on and re-run.")

For most students the **GPU is slower here**. That is the expected result, and it is worth
understanding rather than dismissing.

A GPU is a throughput device. Each of our 3000 epochs launches a handful of tiny kernels on a
400×1 tensor; the fixed cost of launching a kernel (a few tens of microseconds) dwarfs the
arithmetic. The GPU spends its time waiting for work.

"Use a GPU" is not a performance strategy. It pays off when the arithmetic per launch is large:
big batches, wide layers, convolutions. We will hit that point in Task 4.

---
## Part 8 — Save your work somewhere permanent

`/content/` is temporary. Google Drive is not. Mounting Drive asks for permission and then makes
your Drive appear as an ordinary folder at `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, json

OUT = "/content/drive/MyDrive/elec_e8131_colab/task1"
os.makedirs(OUT, exist_ok=True)

# The weights alone are useless. Save the preprocessing constants with them.
torch.save({
    "state_dict": model_log.state_dict(),
    "width": 32,
    "depth": 2,
    "x_mean": float(V.mean()), "x_std": float(V.std()),
    "y_mean": float(y_mean), "y_std": float(y_std),
    "target": "log10_I",
}, os.path.join(OUT, "diode_mlp.pt"))

df.to_csv(os.path.join(OUT, "diode_iv.csv"), index=False)

plt.figure(figsize=(6, 4))
plt.semilogy(V, I, ".", ms=3, label="measured")
plt.semilogy(V, I_pred, "-", lw=2, label="MLP")
plt.xlabel("V [V]"); plt.ylabel("I [A]"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(os.path.join(OUT, "diode_fit.png"), dpi=150, bbox_inches="tight")
plt.close()

print("saved to", OUT)
print(os.listdir(OUT))

---
## Hand-in

Share this notebook (**Share → General access → Anyone with the link → Viewer**) and submit the
link, with your answers written into text cells.

1. In Part 0, what exactly does *Restart session* destroy, and what does it leave alone?
2. Give the physical reason the linear fit cannot work, in one sentence.
3. Attempt 1 reached a *lower* numerical MSE than the model in Part 5 that you would call correct.
   Explain how both statements can be true.
4. State the width you would ship, and justify it with the numbers from Part 6.
5. If you saved only `state_dict` and not the scaling constants, what would go wrong when a
   colleague loads your model next week?

### Optional extension
The Shockley equation depends on temperature through $V_T = kT/q$ and through $I_S$. Generate
curves at 250 K, 300 K and 350 K, train a network with **two** inputs $(V, T)$ on all three at
once, then ask it to predict the curve at 325 K, which it never saw. Does it interpolate sensibly?
This is the smallest useful example of a model that generalises across an operating condition
rather than just along one axis.